In [1]:
import subprocess
import time
from pathlib import Path

import pandas as pd

GENERATED_ROOT = Path("/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations/generated/UltraEdit_Region_10")
SCRIPT_DIR = Path("/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations")
RESULT_CSV = GENERATED_ROOT / "id_to_metrics_ultraeditregion10.csv"
MAX_SAMPLES = 10
GPU = 7

print(f"Result CSV:  {RESULT_CSV}")
print(f"Max samples: {MAX_SAMPLES}")
print(f"GPU:         {GPU}")

Result CSV:  /data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations/generated/UltraEdit_Region_10/id_to_metrics_ultraeditregion10.csv
Max samples: 10
GPU:         7


### Run original `grid_eval.py.bak`

In [4]:
bak_t0 = time.perf_counter()
result_bak = subprocess.run(
    ["python", "grid_eval.py.bak", "--generated-root", str(GENERATED_ROOT), "--gpu", str(GPU)],
    cwd=str(SCRIPT_DIR),
    text=True,
)
elapsed_bak = time.perf_counter() - bak_t0

print(result_bak.stderr)
if result_bak.returncode != 0:
    raise RuntimeError(f"Original run failed (exit {result_bak.returncode})")

df_bak = pd.read_csv(RESULT_CSV)
print(f"Elapsed: {elapsed_bak:.2f}s  ({len(df_bak)} rows)")

Traceback (most recent call last):
  File "/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations/grid_eval.py.bak", line 21, in <module>
    from evaluation.evaluate import calculate_metric
  File "/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations/evaluation/evaluate.py", line 7, in <module>
    from evaluation.matrics_calculator import MetricsCalculator
  File "/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations/evaluation/matrics_calculator.py", line 6, in <module>
    from torchmetrics.multimodal import CLIPScore
  File "/data/home/mirick/miniconda3/envs/pie_eval/lib/python3.9/site-packages/torchmetrics/__init__.py", line 31, in <module>
    import scipy.signal
  File "/data/home/mirick/miniconda3/envs/pie_eval/lib/python3.9/site-packages/scipy/signal/__init__.py", line 330, in <module>
    from ._peak_finding import *
  File "/data/home/mirick/miniconda3/envs/pie_eval/lib/python3.9/site-packages/sci

KeyboardInterrupt: 

### Run optimized `grid_eval.py`

In [5]:
opt_t0 = time.perf_counter()
result_opt = subprocess.run(
    ["python", "grid_eval.py", "--generated-root", str(GENERATED_ROOT), "--gpu", str(GPU)],
    cwd=str(SCRIPT_DIR),
    text=True,
)
elapsed_opt = time.perf_counter() - opt_t0

print(result_opt.stderr)
if result_opt.returncode != 0:
    raise RuntimeError(f"Optimized run failed (exit {result_opt.returncode})")

df_opt = pd.read_csv(RESULT_CSV)
print(f"Elapsed: {elapsed_opt:.2f}s  ({len(df_opt)} rows)")

2026-07-15 12:49:42,944 | INFO | grid_eval | Found 10 samples and 1210 total cells
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Using cache found in /data/home/mirick/.cache/torch/hub/facebookresearch_dino_main
Traceback (most recent call last):
  File "/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations/grid_eval.py", line 296, in <module>
    main()
  File "/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations/grid_eval.py", line 156, in main
    mask_image = Image.open(sample_meta["mask_image_path"])
  File "/data/home/mirick/miniconda3/envs/pie_eval/lib/python3.9/site-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
FileNo

None


RuntimeError: Optimized run failed (exit 1)

## Compare outputs

In [ ]:
print(f"Optimized rows: {len(df_opt)}")
print(f"Original  rows: {len(df_bak)}")

key_cols = ["sample_id", "t_start", "t_end"]
metric_cols = ["psnr_unedit_part", "lpips_unedit_part", "clip_similarity_target_image_edit_part"]

df_merged = df_opt.merge(df_bak, on=key_cols, suffixes=("_opt", "_bak"))
print(f"Matched rows:   {len(df_merged)}")

# Per-metric absolute differences
for col in metric_cols:
    opt_vals = pd.to_numeric(df_merged[f"{col}_opt"], errors="coerce")
    bak_vals = pd.to_numeric(df_merged[f"{col}_bak"], errors="coerce")
    diff = (opt_vals - bak_vals).abs()
    print(f"\n--- {col} ---")
    print(f"  max  abs diff: {diff.max():.8f}")
    print(f"  mean abs diff: {diff.mean():.8f}")
    print(f"  rows with >0.01 diff: {(diff > 0.01).sum()}")

## Timing summary

In [ ]:
speedup = elapsed_bak / elapsed_opt if elapsed_opt > 0 else float("inf")

summary = pd.DataFrame({
    "version": ["original (grid_eval.py.bak)", "optimized (grid_eval.py)"],
    "elapsed_s": [f"{elapsed_bak:.2f}", f"{elapsed_opt:.2f}"],
    "rows": [len(df_bak), len(df_opt)],
})
display(summary)
print(f"\nSpeedup: {speedup:.2f}x")

In [ ]:
import matplotlib.pyplot as plt

n_samples = len(df_opt["sample_id"].unique())
cells_per_sample = len(df_opt) // n_samples

import subprocess as _sp
_short_hash = _sp.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=str(SCRIPT_DIR)).decode().strip()
labels = ["grid_eval (original)", f"grid_eval ({_short_hash})"]
times_s = [elapsed_bak, elapsed_opt]
colors = ["#FF7F0E", "#55A868"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, times_s, color=colors, width=0.55)
ax.set_ylabel("Wall time (s)")
ax.set_title(f"Grid eval runtime ({n_samples} images, {cells_per_sample} cells/image)")
ax.bar_label(bars, labels=[f"{t:.1f}s" for t in times_s], padding=4)
ax.text(
    0.5, 0.92, f"{speedup:.2f}x faster",
    transform=ax.transAxes, ha="center", fontsize=10,
)
ax.set_ylim(0, max(times_s) * 1.15)
fig.tight_layout()
plt.show()